<a id="selecao-modelo"></a>

# TC3-63 — Comparar, avaliar generalização e selecionar o modelo

Consolida os três candidatos desenvolvidos em paralelo na Fase 6:

1. Regressão Logística;
2. Random Forest;
3. Gradient Boosting.

Este notebook **não treina modelos e não acessa o teste**. Ele verifica a equivalência metodológica, consolida resultados e produz uma recomendação transparente para a próxima etapa.


## Inicialização independente

Esta célula reproduz a configuração compartilhada do notebook principal. Ela prepara ambiente, caminhos e funções comuns sem alterar a lógica canônica desta etapa.


In [1]:
# =============================================================================
# CONFIGURAÇÕES PARA RODAR O NOTEBOOK
# Execute esta célula primeiro. Esta política não altera a metodologia do fluxo.
# =============================================================================
import os
import sys
import gc
import ctypes
import polars as pl
from datetime import datetime, timezone

# Opções do usuário
PERFIL_RECURSOS = "auto"       # auto | baixo | medio | normal | alto
LIMITE_RAM_PERCENTUAL = None    # None = automático; ou inteiro entre 50 e 90
MAX_THREADS = None              # None = automático; ou inteiro >= 1
USAR_GPU = os.getenv("TC3_USAR_GPU", "1").strip().lower() in {"1", "true", "sim", "yes"}                 # requer WSL2 + RAPIDS/cuML + kernel tc3-gpu

REUTILIZAR_ARTEFATOS = False  # False reconstrói; True aceita caches válidos
SALVAR_ARTEFATOS = True       # False usa somente área temporária da sessão


# Protocolo metodologico compartilhado por todas as fases e modelos.
SEED_GLOBAL = 42
EXECUTAR_SHAP = os.getenv("TC3_EXECUTAR_SHAP", "0").strip().lower() in {"1", "true", "sim", "yes"}
AMOSTRA_SHAP = max(20, int(os.getenv("TC3_AMOSTRA_SHAP", "100")))
AMOSTRA_BACKGROUND_SHAP = max(20, int(os.getenv("TC3_BACKGROUND_SHAP", "50")))
CHAVE_GRUPO_GLOBAL = "chave_escola_ano"
N_SPLITS_CV_GLOBAL = 3
N_ITER_BUSCA_GLOBAL = 10
LIMIAR_CLASSIFICACAO_GLOBAL = 0.50
REPETICOES_PERMUTACAO_GLOBAL = 5
REPETICOES_BOOTSTRAP_GLOBAL = 200

N_JOBS_MODELAGEM_GLOBAL = 1
N_SPLITS_HOLDOUT_GLOBAL = 1
SCORING_PRINCIPAL_GLOBAL = "roc_auc"
VERBOSE_BUSCA_GLOBAL = 2
ERRO_BUSCA_GLOBAL = "raise"
RETORNAR_SCORE_TREINO_GLOBAL = True

# Contrato comum de pre-processamento dos modelos.
IMPUTACAO_NUMERICA_GLOBAL = "median"
IMPUTACAO_CATEGORICA_GLOBAL = "most_frequent"
ONE_HOT_DESCONHECIDO_GLOBAL = "ignore"
ONE_HOT_SPARSO_GLOBAL = False  # matriz densa: evita fallback dos modelos para CPU
SPARSE_THRESHOLD_GLOBAL = 0.0
ESCALONADOR_CENTRALIZAR_GLOBAL = False


# Percentuais de amostragem (0 < valor <= 1). Alterar aqui afeta todo o notebook.
# O teste final permanece integral (100%); as amostras menores servem apenas ao
# desenvolvimento, comparacao de modelos e interpretabilidade.
FRACOES_AMOSTRAGEM = {
    "eda_correlacao": 0.05,
    "validacao_preparacao": 0.02,
    "fase5_modelos": 0.10,
    "fase5_knn": 0.02,  # KNN tem custo de memoria e predicao maior.
    "fase6_desenvolvimento": 0.12,
    "fase6_validacao_interna": 0.25,
    "fase7_desenvolvimento": 1.00,
    "fase7_teste": 1.00,
    "interpretacao": 0.035,
    "bootstrap": 0.17,
}

for _nome, _fracao in FRACOES_AMOSTRAGEM.items():
    if not 0 < _fracao <= 1:
        raise ValueError(f"Fracao invalida em FRACOES_AMOSTRAGEM[{_nome!r}]: {_fracao}")


def tamanho_por_fracao(total, fracao):
    """Converte uma proporcao em quantidade reproduzivel, sem exceder o total."""
    if total < 1:
        raise ValueError("Nao e possivel amostrar uma particao vazia.")
    return min(total, max(1, round(total * fracao)))


def materializar_split_percentual(dados, contrato, nome_split, fracao, seed, materializador):
    """Materializa uma fracao de um split e informa claramente o universo usado."""
    import polars as pl

    coluna_split = contrato["controle_split"]
    total = (
        dados.filter(pl.col(coluna_split) == nome_split)
        .select(pl.len().alias("total"))
        .collect()
        .item()
    )
    quantidade = tamanho_por_fracao(total, fracao)
    X, y = materializador(dados, contrato, nome_split, limit=quantidade, seed=seed)
    print(f"{nome_split}: {quantidade:,}/{total:,} registros ({fracao:.1%})")
    return X, y


def _estado_memoria_sistema():
    """Retorna RAM total e disponível em bytes."""
    valores = {}
    try:
        with open("/proc/meminfo", encoding="utf-8") as arquivo:
            for linha in arquivo:
                chave, valor = linha.split(":", 1)
                valores[chave] = int(valor.strip().split()[0]) * 1024
        return valores["MemTotal"], valores.get("MemAvailable", valores.get("MemFree", 0))
    except (OSError, KeyError, ValueError):
        try:
            return (
                os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE"),
                os.sysconf("SC_AVPHYS_PAGES") * os.sysconf("SC_PAGE_SIZE"),
            )
        except (AttributeError, OSError, ValueError):
            return None, None


RAM_TOTAL_BYTES, RAM_DISPONIVEL_BYTES = _estado_memoria_sistema()
RAM_TOTAL_GB = RAM_TOTAL_BYTES / 1024**3 if RAM_TOTAL_BYTES else 0.0
CPUS_LOGICAS = os.cpu_count() or 1

if RAM_TOTAL_GB and RAM_TOTAL_GB <= 4.5:
    _perfil_detectado = "baixo"
elif RAM_TOTAL_GB and RAM_TOTAL_GB <= 8.5:
    _perfil_detectado = "medio"
elif RAM_TOTAL_GB and RAM_TOTAL_GB <= 16.5:
    _perfil_detectado = "normal"
else:
    _perfil_detectado = "alto"

_perfis = {
    "baixo": (60, 1),
    "medio": (70, min(2, CPUS_LOGICAS)),
    "normal": (75, min(4, CPUS_LOGICAS)),
    "alto": (80, min(8, max(1, CPUS_LOGICAS // 2))),
}
PERFIL_ATIVO = _perfil_detectado if PERFIL_RECURSOS == "auto" else PERFIL_RECURSOS.lower()
if PERFIL_ATIVO not in _perfis:
    raise ValueError("PERFIL_RECURSOS deve ser auto, baixo, medio, normal ou alto.")

_limite_perfil, _threads_perfil = _perfis[PERFIL_ATIVO]
LIMITE_RAM_ATIVO = _limite_perfil if LIMITE_RAM_PERCENTUAL is None else int(LIMITE_RAM_PERCENTUAL)
THREADS_MAXIMAS = _threads_perfil if MAX_THREADS is None else int(MAX_THREADS)
if not 50 <= LIMITE_RAM_ATIVO <= 90:
    raise ValueError("LIMITE_RAM_PERCENTUAL deve ficar entre 50 e 90.")
if THREADS_MAXIMAS < 1:
    raise ValueError("MAX_THREADS deve ser pelo menos 1.")
THREADS_MAXIMAS = min(THREADS_MAXIMAS, CPUS_LOGICAS)

# Definido antes de NumPy, Polars, scikit-learn e cuML serem importados.
for _variavel, _valor in {
    "OMP_NUM_THREADS": str(THREADS_MAXIMAS),
    "OPENBLAS_NUM_THREADS": str(THREADS_MAXIMAS),
    "MKL_NUM_THREADS": str(THREADS_MAXIMAS),
    "NUMEXPR_NUM_THREADS": str(THREADS_MAXIMAS),
    "VECLIB_MAXIMUM_THREADS": str(THREADS_MAXIMAS),
    "POLARS_MAX_THREADS": str(THREADS_MAXIMAS),
    "MALLOC_ARENA_MAX": "2",
}.items():
    os.environ[_variavel] = _valor

HISTORICO_RECURSOS = []
_estado_recursos = {"nivel": None, "threads": None, "contexto_threads": None}


def _aplicar_threads(limite):
    """Limita pools BLAS/OpenMP já carregados; Polars usa o limite inicial."""
    limite = max(1, min(int(limite), THREADS_MAXIMAS))
    if _estado_recursos["threads"] == limite:
        return
    anterior = _estado_recursos.get("contexto_threads")
    if anterior is not None:
        try:
            anterior.__exit__(None, None, None)
        except Exception:
            pass
    try:
        from threadpoolctl import threadpool_limits
        contexto = threadpool_limits(limits=limite)
        contexto.__enter__()
    except Exception:
        contexto = None
    _estado_recursos["contexto_threads"] = contexto
    _estado_recursos["threads"] = limite


def _liberar_memoria_global(agressiva=False):
    """Libera caches transitórios sem apagar variáveis usadas pelo fluxo."""
    shell = get_ipython()
    if shell is not None:
        for nome in ("_", "__", "___"):
            shell.user_ns[nome] = None
    gc.collect()
    modulo_cupy = sys.modules.get("cupy")
    if modulo_cupy is not None:
        try:
            modulo_cupy.get_default_memory_pool().free_all_blocks()
            modulo_cupy.get_default_pinned_memory_pool().free_all_blocks()
        except Exception:
            pass
    if agressiva:
        modulo_plt = sys.modules.get("matplotlib.pyplot")
        if modulo_plt is not None:
            try:
                modulo_plt.close("all")
            except Exception:
                pass
        try:
            ctypes.CDLL("libc.so.6").malloc_trim(0)
        except (OSError, AttributeError):
            pass


def _gerenciar_recursos(*_args, **_kwargs):
    """Reduz paralelismo progressivamente e nunca interrompe uma célula."""
    total, disponivel = _estado_memoria_sistema()
    if not total:
        return
    uso = 100.0 * (total - disponivel) / total
    margem = LIMITE_RAM_ATIVO - uso
    if margem > 15:
        nivel, threads, limpar = "normal", THREADS_MAXIMAS, False
    elif margem > 7:
        nivel, threads, limpar = "moderado", max(1, THREADS_MAXIMAS // 2), True
    else:
        nivel, threads, limpar = "conservador", 1, True

    _aplicar_threads(threads)
    if limpar:
        _liberar_memoria_global(agressiva=margem <= 0)

    if nivel != _estado_recursos["nivel"]:
        registro = {
            "instante_utc": datetime.now(timezone.utc).isoformat(),
            "nivel": nivel,
            "threads": threads,
            "ram_usada_percentual": round(uso, 1),
            "ram_disponivel_gb": round(disponivel / 1024**3, 2),
        }
        HISTORICO_RECURSOS.append(registro)
        print(
            f"[recursos] nível={nivel}; threads={threads}; "
            f"RAM usada={uso:.1f}%; livre={disponivel / 1024**3:.1f} GB."
        )
        _estado_recursos["nivel"] = nivel


ipython = get_ipython()
if ipython is not None:
    # Impede callbacks duplicados ao reexecutar a configuração.
    for _atributo in ("_tc3_memory_guard", "_tc3_resource_guard"):
        _guard_anterior = getattr(ipython, _atributo, None)
        if _guard_anterior is not None:
            for _evento, _funcao in _guard_anterior:
                try:
                    ipython.events.unregister(_evento, _funcao)
                except (ValueError, KeyError):
                    pass
    _callbacks_recursos = [
        ("pre_run_cell", _gerenciar_recursos),
        ("post_run_cell", _gerenciar_recursos),
    ]
    for _evento, _funcao in _callbacks_recursos:
        ipython.events.register(_evento, _funcao)
    ipython._tc3_resource_guard = _callbacks_recursos

_gerenciar_recursos()
print(
    "Configuração automática ativa: "
    f"perfil={PERFIL_ATIVO}; RAM={RAM_TOTAL_GB:.1f} GB; CPUs={CPUS_LOGICAS}; "
    f"limite preventivo={LIMITE_RAM_ATIVO}%; threads máximas={THREADS_MAXIMAS}."
)

# Backend GPU: ativado antes de qualquer importação do scikit-learn.
cuda_conda = os.path.join(sys.prefix, "targets", "x86_64-linux")
if os.path.exists(os.path.join(cuda_conda, "include", "cuda.h")):
    os.environ.setdefault("CUDA_PATH", cuda_conda)
if USAR_GPU:
    try:
        from cuml.accel import install
        install(log_level="info")
        import cupy as cp
        dispositivo = cp.cuda.runtime.getDeviceProperties(0)
        nome_gpu = dispositivo["name"].decode() if isinstance(dispositivo["name"], bytes) else str(dispositivo["name"])
        memoria_gpu_gb = cp.cuda.runtime.memGetInfo()[1] / 1024**3
        BACKEND_ML = f"RAPIDS cuML — {nome_gpu} ({memoria_gpu_gb:.1f} GB de VRAM)"
    except Exception as exc:
        raise RuntimeError("GPU/RAPIDS indisponível. Abra no WSL2 e selecione o kernel tc3-gpu.") from exc
else:
    BACKEND_ML = "scikit-learn CPU"
print("Backend de Machine Learning:", BACKEND_ML)

from pathlib import Path
import hashlib
import json
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

def localizar_raiz(inicio: Path) -> Path:
    for candidato in [inicio, *inicio.parents]:
        if (candidato / "dados").is_dir() and (candidato / "notebooks").is_dir():
            return candidato
    raise FileNotFoundError("Execute este notebook dentro do repositório tech_challenge_3.")

PROJECT_ROOT = localizar_raiz(Path.cwd().resolve())
print("Raiz do projeto:", PROJECT_ROOT)


# Contrato único de diretórios. Fontes sempre são lidas da área imutável.
import tempfile

FONTES_ORIGINAIS_DIR = PROJECT_ROOT / "dados" / "fontes_originais"
_TEMP_BASE = PROJECT_ROOT / ".execucao_temporaria"
_TEMP_BASE.mkdir(parents=True, exist_ok=True)
if SALVAR_ARTEFATOS:
    RAIZ_ARTEFATOS = PROJECT_ROOT
    _DIRETORIO_TEMPORARIO = None
else:
    _DIRETORIO_TEMPORARIO = tempfile.TemporaryDirectory(prefix="tc3_", dir=_TEMP_BASE)
    RAIZ_ARTEFATOS = Path(_DIRETORIO_TEMPORARIO.name)

DADOS_PROCESSADOS_DIR = RAIZ_ARTEFATOS / "dados" / "dados_processados"
DADOS_ANALITICOS_DIR = RAIZ_ARTEFATOS / "dados" / "dados_analiticos"
RESULTADOS_DIR = RAIZ_ARTEFATOS / "resultados"
RESULTADOS_FIGURAS_DIR = RESULTADOS_DIR / "figuras"
RESULTADOS_TABELAS_DIR = RESULTADOS_DIR / "tabelas"
RESULTADOS_METRICAS_DIR = RESULTADOS_DIR / "metricas"
RESULTADOS_RELATORIOS_DIR = RESULTADOS_DIR / "relatorios"
MODELOS_DIR = RAIZ_ARTEFATOS / "modelos"

for _diretorio_saida in (
    DADOS_PROCESSADOS_DIR, DADOS_ANALITICOS_DIR, RESULTADOS_FIGURAS_DIR,
    RESULTADOS_TABELAS_DIR, RESULTADOS_METRICAS_DIR,
    RESULTADOS_RELATORIOS_DIR, MODELOS_DIR,
):
    _diretorio_saida.mkdir(parents=True, exist_ok=True)

def deve_recalcular(caminho):
    """True quando o cache deve ser ignorado ou ainda não existe."""
    return not REUTILIZAR_ARTEFATOS or not Path(caminho).exists()


def publicar_parquet_particionado(frame, destino, *, particionar_por):
    """Publica partições sequencialmente e troca o diretório de forma atômica.

    O ``write_parquet(partition_by=...)`` do Polars prepara várias partições em
    paralelo e pode duplicar o pico de RAM de bases já materializadas. Esta
    implementação mantém apenas uma partição adicional em memória por vez.
    """
    import gc
    import shutil

    destino = Path(destino)
    temporario = destino.with_name(destino.name + "__publicacao_tmp")
    anterior = destino.with_name(destino.name + "__publicacao_anterior")
    for caminho in (temporario, anterior):
        if caminho.exists():
            shutil.rmtree(caminho)
    temporario.mkdir(parents=True, exist_ok=True)

    lazy = frame if isinstance(frame, pl.LazyFrame) else frame.lazy()
    schema = lazy.collect_schema()
    if particionar_por not in schema.names():
        raise ValueError(f"Coluna de particionamento ausente: {particionar_por}")

    valores = (
        lazy.select(pl.col(particionar_por).unique())
        .collect(engine="streaming")
        .get_column(particionar_por)
        .to_list()
    )
    valores = sorted(valores, key=lambda valor: (valor is None, str(valor)))
    if not valores:
        raise RuntimeError("Não há valores para particionar a publicação.")

    arquivos_novos = []
    for indice, valor in enumerate(valores):
        valor_hive = "__HIVE_DEFAULT_PARTITION__" if valor is None else str(valor)
        if any(separador in valor_hive for separador in ("/", "\\")):
            raise ValueError(f"Valor de partição inválido: {valor_hive!r}")
        diretorio_particao = temporario / f"{particionar_por}={valor_hive}"
        diretorio_particao.mkdir(parents=True, exist_ok=True)
        arquivo = diretorio_particao / f"part-{indice:05d}.parquet"
        predicado = pl.col(particionar_por).is_null() if valor is None else pl.col(particionar_por) == pl.lit(valor)
        lazy.filter(predicado).sink_parquet(
            arquivo,
            compression="zstd",
            statistics=True,
            row_group_size=100_000,
            mkdir=True,
            engine="streaming",
        )
        if not arquivo.is_file() or arquivo.stat().st_size == 0:
            raise RuntimeError(f"Partição não foi gravada corretamente: {arquivo}")
        arquivos_novos.append(arquivo)
        gc.collect()

    if destino.exists():
        destino.replace(anterior)
    try:
        try:
            temporario.replace(destino)
        except OSError:
            # Volumes Docker montados do Windows podem bloquear rename de diretórios.
            # O fallback copia a publicação pronta e remove a área temporária.
            shutil.copytree(temporario, destino)
            shutil.rmtree(temporario)
    except Exception:
        if destino.exists():
            shutil.rmtree(destino)
        if anterior.exists():
            anterior.replace(destino)
        raise
    if anterior.exists():
        shutil.rmtree(anterior)
    return len(arquivos_novos)


def salvar_figura_eda(figura, nome):
    """Exporta uma figura exploratória sem alterar sua exibição no notebook."""
    destino = RESULTADOS_FIGURAS_DIR / "fluxo_exploratorio" / nome
    destino.parent.mkdir(parents=True, exist_ok=True)
    figura.savefig(destino, dpi=180, bbox_inches="tight")
    return destino

print("Fontes originais:", FONTES_ORIGINAIS_DIR)
print("Destino dos artefatos:", RAIZ_ARTEFATOS)
print("Reutilizar artefatos:", REUTILIZAR_ARTEFATOS, "| Salvar:", SALVAR_ARTEFATOS)

# Imports compartilhados que, no fluxo completo, são carregados por etapas anteriores.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display


[recursos] nível=normal; threads=4; RAM usada=30.8%; livre=10.7 GB.
Configuração automática ativa: perfil=normal; RAM=15.5 GB; CPUs=20; limite preventivo=75%; threads máximas=4.
[cuml.accel] Accelerator installed.
Backend de Machine Learning: RAPIDS cuML — NVIDIA GeForce RTX 3050 6GB Laptop GPU (6.0 GB de VRAM)
Raiz do projeto: /workspace
Fontes originais: /workspace/dados/fontes_originais
Destino dos artefatos: /workspace
Reutilizar artefatos: False | Salvar: True


In [2]:
from pathlib import Path
import json

# PROJECT_ROOT e herdado da configuracao central.
RESULT_DIR = RESULTADOS_DIR / "metricas" / "fase_6"
EXPECTED = {
    "TC3-60": RESULT_DIR / "TC3-60_resultado_otimizacao.json",
    "TC3-61": RESULT_DIR / "TC3-61_resultado_otimizacao.json",
    "TC3-62": RESULT_DIR / "TC3-62_resultado_otimizacao.json",
}

faltantes = [ticket for ticket, path in EXPECTED.items() if not path.exists()]
if faltantes:
    raise FileNotFoundError(f"Execute primeiro os notebooks: {faltantes}")

resultados = [json.loads(path.read_text(encoding="utf-8")) for path in EXPECTED.values()]
print("Resultados carregados:", len(resultados))


Resultados carregados: 3


## 1. Validação do protocolo comum


In [3]:
campos_comuns = [
    "manifesto", "features", "chave_agrupamento", "amostra_desenvolvimento",
    "registros_treino", "registros_validacao", "n_splits_cv", "seed",
]
referencia = resultados[0]["protocolo"]
for resultado in resultados:
    protocolo = resultado["protocolo"]
    divergencias = [campo for campo in campos_comuns if protocolo[campo] != referencia[campo]]
    assert not divergencias, f"{resultado['modelo']['id']} diverge em: {divergencias}"
    assert protocolo["teste_utilizado"] is False
    assert protocolo["sobreposicao_grupos"] == 0

print("Protocolos equivalentes: sim")
print("Teste utilizado: não")
print("Sobreposição de grupos: zero")


Protocolos equivalentes: sim
Teste utilizado: não
Sobreposição de grupos: zero


## 2. Ranking dos candidatos


In [4]:
linhas = [{
    "ticket": r["modelo"]["id"], "modelo": r["modelo"]["nome"],
    "roc_auc_cv": r["busca"]["roc_auc_cv_media"],
    "desvio_auc_cv": r["busca"]["roc_auc_cv_desvio"],
    "roc_auc_validacao": r["metricas_validacao"]["roc_auc"],
    "average_precision": r["metricas_validacao"]["average_precision"],
    "balanced_accuracy": r["metricas_validacao"]["balanced_accuracy"],
    "f1": r["metricas_validacao"]["f1"], "gap_roc_auc": r["gap_roc_auc"],
    "tempo_busca_s": r["busca"]["tempo_total_segundos"],
} for r in resultados]
ranking = (
    pl.DataFrame(linhas)
    .sort(["roc_auc_validacao", "average_precision", "desvio_auc_cv", "gap_roc_auc"],
          descending=[True, True, False, False])
    .with_row_index("posicao", offset=1)
)
display(ranking)


posicao,ticket,modelo,roc_auc_cv,desvio_auc_cv,roc_auc_validacao,average_precision,balanced_accuracy,f1,gap_roc_auc,tempo_busca_s
u32,str,str,f64,f64,f64,f64,f64,f64,f64,f64
1,"""TC3-61""","""Random Forest""",0.657314,0.001819,0.656575,0.665661,0.610681,0.634245,0.025674,259.408837
2,"""TC3-62""","""Gradient Boosting""",0.65631,0.001802,0.656137,0.665751,0.612082,0.624977,0.028564,91.771017
3,"""TC3-60""","""Regressão Logística""",0.605605,0.010646,0.59206,0.599088,0.561083,0.57468,0.002001,64.057625


## 3. Regra de decisão

A recomendação automática usa ROC AUC de validação como critério primário. Contudo, diferenças inferiores a `0,005` são tratadas como empate técnico: nesse caso, estabilidade, average precision, gap, interpretabilidade e custo devem orientar a decisão do grupo.


In [5]:
melhor = ranking.row(0, named=True)
segundo = ranking.row(1, named=True)
diferenca_auc = float(melhor["roc_auc_validacao"] - segundo["roc_auc_validacao"])
empate_tecnico = diferenca_auc < 0.005

MODELO_SELECIONADO = "Random Forest"
TICKET_SELECIONADO = "TC3-61"
selecionado = ranking.filter(pl.col("ticket") == TICKET_SELECIONADO).row(0, named=True)

if not empate_tecnico:
    raise RuntimeError(
        "A decisão colegiada pelo Random Forest pressupõe empate técnico; revise os resultados antes de prosseguir."
    )

justificativas_selecao = [
    "Gradient Boosting e Random Forest ficaram dentro do limite pré-definido de empate técnico de 0,005 no holdout de validação.",
    "O Random Forest apresentou a maior média de ROC AUC na validação cruzada agrupada.",
    "O Random Forest apresentou a maior média de ROC AUC na validação cruzada agrupada, menor gap entre treino e validação e F1 ligeiramente superior ao Gradient Boosting.",
    "A decisão foi colegiada e registrada antes de qualquer acesso ao conjunto de teste.",
]

print("Líder no holdout de validação:", melhor["modelo"])
print("Diferença para o segundo:", diferenca_auc)
print("Empate técnico: sim")
print("Modelo selecionado pelo grupo:", MODELO_SELECIONADO)
print("AUC CV do selecionado:", float(selecionado["roc_auc_cv"]))
print("Desvio AUC CV do selecionado:", float(selecionado["desvio_auc_cv"]))
for justificativa in justificativas_selecao:
    print("-", justificativa)


Líder no holdout de validação: Random Forest
Diferença para o segundo: 0.0004383568621313394
Empate técnico: sim
Modelo selecionado pelo grupo: Random Forest
AUC CV do selecionado: 0.6573144864150683
Desvio AUC CV do selecionado: 0.0018193028137203555
- Gradient Boosting e Random Forest ficaram dentro do limite pré-definido de empate técnico de 0,005 no holdout de validação.
- O Random Forest apresentou a maior média de ROC AUC na validação cruzada agrupada.
- O Random Forest apresentou a maior média de ROC AUC na validação cruzada agrupada, menor gap entre treino e validação e F1 ligeiramente superior ao Gradient Boosting.
- A decisão foi colegiada e registrada antes de qualquer acesso ao conjunto de teste.


## 4. Exportação da consolidação


In [6]:
CONSOLIDATED_CSV = RESULT_DIR / "TC3-63_ranking_modelos.csv"
CONSOLIDATED_JSON = RESULT_DIR / "TC3-63_selecao_modelo.json"

ranking.write_csv(CONSOLIDATED_CSV)
consolidado = {
    "schema_resultado": 2,
    "fase": 6,
    "teste_utilizado": False,
    "criterio_primario": "roc_auc_validacao",
    "limiar_empate_tecnico": 0.005,
    "empate_tecnico": empate_tecnico,
    "lider_holdout_validacao": melhor["modelo"],
    "diferenca_auc_primeiro_segundo": diferenca_auc,
    "modelo_selecionado": MODELO_SELECIONADO,
    "ticket_selecionado": TICKET_SELECIONADO,
    "tipo_decisao": "decisão colegiada entre modelos tecnicamente empatados",
    "momento_decisao": "antes do acesso ao conjunto de teste",
    "justificativas_selecao": justificativas_selecao,
    "metricas_modelo_selecionado": dict(selecionado),
    "ranking": ranking.to_dicts(),
    "proxima_etapa": "avaliar exclusivamente o Random Forest no teste reservado",
}
CONSOLIDATED_JSON.write_text(json.dumps(consolidado, ensure_ascii=False, indent=2), encoding="utf-8")
print("Ranking:", CONSOLIDATED_CSV)
print("Decisão oficial:", CONSOLIDATED_JSON)
print("Modelo oficialmente selecionado:", MODELO_SELECIONADO)


Ranking: /workspace/resultados/metricas/fase_6/TC3-63_ranking_modelos.csv
Decisão oficial: /workspace/resultados/metricas/fase_6/TC3-63_selecao_modelo.json
Modelo oficialmente selecionado: Random Forest


## 5. Decisão oficial e próxima etapa

O grupo selecionou o **Random Forest** dentro do empate técnico observado na Fase 6. A decisão foi tomada sem acesso ao teste e priorizou a maior média de ROC AUC na validação cruzada agrupada o menor gap entre treino e validação e o F1 ligeiramente superior, além da avaliação colegiada do grupo.

Na Fase 7, somente o Random Forest deve:

1. ter seus hiperparâmetros e limiar congelados;
2. ser treinado novamente com todo o conjunto de desenvolvimento;
3. ser avaliado uma única vez no teste reservado;
4. produzir incerteza, calibração e análises territoriais;
5. ser interpretado e relacionado às metas oficiais.

O teste não foi utilizado para desempatar ou escolher modelos.


<a id="fase-7"></a>

# Fase 7 — avaliação independente do Random Forest
